# Train the Fashion Product Analyser on a free GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajayvighnesh/fashion-product-analyser/blob/main/notebooks/train_colab.ipynb)

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

The notebook will:
1. install the project
2. download the dataset (~44k product photos)
3. fine-tune the ViT model on category and colour (about 20–40 minutes on a T4)
4. show the test results, confusion matrices and example predictions
5. run the image + title pipeline and draft Stable Diffusion prompts
6. save the trained model to your Google Drive (optional)

At the end, a **results table** is printed. Copy it into the README.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to T4 GPU first"

## 1. Install

In [ ]:
import os, sys
if not os.path.exists("fashion-product-analyser"):
    !git clone -q https://github.com/ajayvighnesh/fashion-product-analyser
%cd /content/fashion-product-analyser
!pip install -q -e ".[data,demo]"
sys.path.insert(0, "src")  # so the package imports without restarting the runtime

## 2. Download the dataset
From the Hugging Face mirror of *Fashion Product Images (Small)*, no account needed. Takes a few minutes.

In [ ]:
if not os.path.exists("data/styles.csv"):
    !python scripts/download_data.py --out data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from fashion_analyser.data import load_metadata
from fashion_analyser.preprocess import load_image

df = load_metadata("data", top_k_categories=20)
print(f"{len(df)} products, {df['category'].nunique()} categories")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df["category"].value_counts().plot.bar(ax=ax[0], title="Category")
df["colour"].value_counts().plot.bar(ax=ax[1], title="Colour group")
plt.tight_layout(); plt.show()

sample = df.sample(8, random_state=0)
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for a, (_, r) in zip(axes, sample.iterrows()):
    a.imshow(load_image(r["image_path"])); a.axis("off")
    a.set_title(f"{r['category']}\n{r['colour']}", fontsize=8)
plt.show()

## 3. Train
Epoch 1 trains only the two heads (backbone frozen); epochs 2–5 fine-tune everything.
Progress is printed every 200 steps. If Colab disconnects, run this cell again.

In [ ]:
!PYTHONPATH=src python -m fashion_analyser.train \
    --data-dir data --out-dir runs/vit --epochs 5 --freeze-epochs 1 --batch-size 64 --num-workers 2

## 4. Results

In [ ]:
import json
m = json.load(open("runs/vit/metrics.json"))
t = m["test"]
hist = pd.DataFrame(m["history"])
display(hist)

table = f"""| Head | Test accuracy | Test macro-F1 |
|---|---|---|
| Category ({df['category'].nunique()} classes) | {t['category_acc']:.1%} | {t['category_macro_f1']:.3f} |
| Colour (14 groups) | {t['colour_acc']:.1%} | {t['colour_macro_f1']:.3f} |

Backbone `{m['backbone']}`, best epoch {m['best_epoch']} of {len(hist)}, \
{m['n_train']:,} train / {m['n_val']:,} val / {m['n_test']:,} test images."""
print("COPY THIS INTO THE README:\n")
print(table)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(hist["epoch"], hist["val_category_macro_f1"], "o-", label="category")
ax[0].plot(hist["epoch"], hist["val_colour_macro_f1"], "o-", label="colour")
ax[0].set_title("Validation macro-F1"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["epoch"], hist["train_loss"], "o-"); ax[1].set_title("Train loss"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig("runs/vit/training_curves.png", dpi=120); plt.show()

In [ ]:
import numpy as np
def plot_cm(path, title, ax):
    cm = pd.read_csv(path, index_col=0)
    norm = cm.div(cm.sum(1).replace(0, 1), axis=0)  # row-normalised: recall per class
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.index, fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    off = norm.where(~np.eye(len(norm), dtype=bool)).stack().sort_values(ascending=False).head(5)
    return off

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
c1 = plot_cm("runs/vit/confusion_category.csv", "Category (row-normalised)", ax[0])
c2 = plot_cm("runs/vit/confusion_colour.csv", "Colour (row-normalised)", ax[1])
plt.tight_layout(); plt.savefig("runs/vit/confusion_matrices.png", dpi=120); plt.show()
print("Most common category mix-ups (true -> predicted, share of true class):"); print(c1.round(3).to_string())
print("\nMost common colour mix-ups:"); print(c2.round(3).to_string())

## 5. Example predictions on unseen test images
Green title = both heads correct; red = at least one wrong.

In [ ]:
import torch
from fashion_analyser.data import split
from fashion_analyser.predict import Analyser

device = "cuda" if torch.cuda.is_available() else "cpu"
analyser = Analyser.from_checkpoint("runs/vit", device=device)
_, _, test_df = split(df, seed=42)  # same split as training

rows = test_df.sample(12, random_state=1)
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for a, (_, r) in zip(axes.ravel(), rows.iterrows()):
    p = analyser.predict(r["image_path"])
    ok = p.category == r["category"] and p.colour == r["colour"]
    a.imshow(load_image(r["image_path"])); a.axis("off")
    a.set_title(f"pred: {p.colour} {p.category} ({p.category_confidence:.0%})\n"
                f"true: {r['colour']} {r['category']}", fontsize=8, color="green" if ok else "red")
plt.tight_layout(); plt.savefig("runs/vit/example_predictions.png", dpi=120); plt.show()

## 6. Image + title pipeline and design-variant prompts
The same tools the LangChain agent uses, run without an LLM.

In [ ]:
from fashion_analyser.agent import analyse_product
from fashion_analyser.prompts import draft_prompt

for _, r in test_df.sample(5, random_state=3).iterrows():
    rec = analyse_product(analyser, r["image_path"], r["productDisplayName"])
    print(f"Title: {r['productDisplayName']}")
    print(f"  -> {rec['colour']} {rec['category']} | gender: {rec['gender']} | needs review: {rec['needs_review']}")
    for issue in rec["conflicts"] + rec["warnings"]:
        print(f"     ! {issue}")
    print(f"  SD prompt: {draft_prompt(rec['category'], rec['colour'], rec['gender'], 'pastel colours, summer collection').prompt[:110]}...\n")

In [ ]:
# How often do the photo and the title disagree on the test set? (useful data-quality finding)
from tqdm.auto import tqdm
recs = [analyse_product(analyser, r["image_path"], r["productDisplayName"])
        for _, r in tqdm(test_df.iterrows(), total=len(test_df))]
n = len(recs)
colour_conf = sum(any(c.startswith("colour") for c in x["conflicts"]) for x in recs)
cat_conf = sum(any(c.startswith("category") for c in x["conflicts"]) for x in recs)
review = sum(x["needs_review"] for x in recs)
print(f"Test products: {n}")
print(f"Photo vs title colour conflict:   {colour_conf} ({colour_conf/n:.1%})")
print(f"Photo vs title category conflict: {cat_conf} ({cat_conf/n:.1%})")
print(f"Flagged for human review:         {review} ({review/n:.1%})")

## 7. Save the trained model
Colab deletes files when the session ends. Save to Google Drive, or download the zip.

In [ ]:
SAVE_TO_DRIVE = True
!cd runs && zip -qr vit_run.zip vit
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/fashion-product-analyser
    !cp runs/vit_run.zip /content/drive/MyDrive/fashion-product-analyser/
    print("Saved to My Drive/fashion-product-analyser/vit_run.zip")
else:
    from google.colab import files
    files.download("runs/vit_run.zip")

## 8. Optional: live demo
Starts the Gradio app with a public link that lasts 72 hours, so you can try your own photos or show someone.

In [ ]:
from app import build_ui
build_ui(analyser).launch(share=True)

## Done
Send back the **results table** from step 4, the top mix-ups and the conflict numbers from step 6.
Those go into the README, and the three saved images (`training_curves.png`, `confusion_matrices.png`,
`example_predictions.png`) in `runs/vit/` can be added to `docs/`.